In [ ]:
from dataclasses import dataclass
from typing import Dict, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import brentq
from scipy.stats import norm
import statsmodels.api as sm


@dataclass
class SimulationConfig:
    n_days: int = 1260  # 5 years of daily trading data
    dt: float = 1 / 252
    s0: float = 2.25  # Initial USGC Jet Fuel spot price ($/gal)
    f0: float = 1.95  # Initial liquid proxy futures price ($/gal equivalent)
    mu: float = 0.03  # Annualized drift
    sigma_f: float = 0.34  # Futures annualized volatility
    beta_true: float = 0.82  # Structural cross-hedge pass-through ratio
    kappa: float = 3.5  # Mean-reversion speed of jet crack spread
    theta_crack: float = 0.65  # Long-run jet crack spread ($/gal)
    sigma_crack: float = 0.16  # Volatility of the crack/basis spread
    jump_intensity: float = 2.0  # Expected geopolitical/refinery shocks per year
    jump_mean: float = 0.07  # Mean upward price shock
    jump_std: float = 0.05  # Shock volatility
    risk_free_rate: float = 0.045
    monthly_gallons: float = 10_000_000  # 10M gallons/month airline fuel burn
    seed: int = 42


class MarketDataGenerator:
    """Simulates Jet Fuel spot & liquid proxy futures with crack-spread basis risk and jumps."""

    def __init__(self, cfg: SimulationConfig):
        self.cfg = cfg

    def generate(self) -> pd.DataFrame:
        rng = np.random.default_rng(self.cfg.seed)
        n = self.cfg.n_days
        dt = self.cfg.dt

        futures = np.zeros(n)
        crack = np.zeros(n)
        spot = np.zeros(n)

        futures[0] = self.cfg.f0
        crack[0] = self.cfg.s0 - self.cfg.beta_true * self.cfg.f0
        spot[0] = self.cfg.s0

        for t in range(1, n):
            # Merton Jump-Diffusion for liquid futures proxy
            z_f = rng.standard_normal()
            n_jumps = rng.poisson(self.cfg.jump_intensity * dt)
            jump_factor = (
                np.sum(rng.normal(self.cfg.jump_mean, self.cfg.jump_std, n_jumps))
                if n_jumps > 0
                else 0.0
            )
            futures[t] = futures[t - 1] * np.exp(
                (self.cfg.mu - 0.5 * self.cfg.sigma_f**2) * dt
                + self.cfg.sigma_f * np.sqrt(dt) * z_f
                + jump_factor
            )

            # Ornstein-Uhlenbeck process for Jet Crack Spread (basis risk)
            z_c = rng.standard_normal()
            refinery_shock = 0.4 * jump_factor if n_jumps > 0 else 0.0
            crack[t] = (
                crack[t - 1]
                + self.cfg.kappa * (self.cfg.theta_crack - crack[t - 1]) * dt
                + self.cfg.sigma_crack * np.sqrt(dt) * z_c
                + refinery_shock
            )
            spot[t] = np.maximum(0.60, self.cfg.beta_true * futures[t] + crack[t])

        dates = pd.bdate_range(start="2021-01-04", periods=n)
        df = pd.DataFrame(
            {"Jet_Spot": spot, "Proxy_Futures": futures, "Crack_Spread": spot - futures},
            index=dates,
        )
        df["Ret_Spot"] = np.log(df["Jet_Spot"] / df["Jet_Spot"].shift(1))
        df["Ret_Futures"] = np.log(df["Proxy_Futures"] / df["Proxy_Futures"].shift(1))
        return df.dropna()


class Black76Engine:
    """Analytical Black-76 pricing, Greeks, and Zero-Cost Collar solver for futures options."""

    @staticmethod
    def price_and_greeks(
        F: float, K: float, T: float, r: float, sigma: float, option_type: str = "call"
    ) -> Dict[str, float]:
        if T <= 1e-8:
            intrinsic = max(0.0, F - K) if option_type == "call" else max(0.0, K - F)
            return {"price": intrinsic, "delta": 0.0, "gamma": 0.0, "vega": 0.0}

        vol_sqrt_t = sigma * np.sqrt(T)
        d1 = (np.log(F / K) + 0.5 * sigma**2 * T) / vol_sqrt_t
        d2 = d1 - vol_sqrt_t
        df = np.exp(-r * T)

        if option_type.lower() == "call":
            price = df * (F * norm.cdf(d1) - K * norm.cdf(d2))
            delta = df * norm.cdf(d1)
        elif option_type.lower() == "put":
            price = df * (K * norm.cdf(-d2) - F * norm.cdf(-d1))
            delta = -df * norm.cdf(-d1)
        else:
            raise ValueError("option_type must be 'call' or 'put'")

        gamma = (df * norm.pdf(d1)) / (F * vol_sqrt_t)
        vega = df * F * norm.pdf(d1) * np.sqrt(T)
        return {"price": price, "delta": delta, "gamma": gamma, "vega": vega}

    @classmethod
    def solve_zero_cost_collar(
        cls, F: float, K_call: float, T: float, r: float, sigma: float
    ) -> Tuple[float, float]:
        """Solves for K_put such that Long Call(K_call) - Short Put(K_put) = 0."""
        call_val = cls.price_and_greeks(F, K_call, T, r, sigma, "call")["price"]

        def objective(K_put: float) -> float:
            put_val = cls.price_and_greeks(F, K_put, T, r, sigma, "put")["price"]
            return call_val - put_val

        K_put = brentq(objective, 0.25 * F, K_call)
        return K_put, call_val


class HedgeRatioModels:
    """Econometric estimators for Minimum Variance Hedge Ratio (MVHR)."""

    @staticmethod
    def static_ols_mvhr(ret_s: pd.Series, ret_f: pd.Series) -> Tuple[float, float]:
        """OLS on log-returns: beta = Cov(dlnS, dlnF) / Var(dlnF)."""
        X = sm.add_constant(ret_f)
        model = sm.OLS(ret_s, X).fit()
        return float(model.params.iloc[1]), float(model.rsquared)

    @staticmethod
    def ewma_mvhr(
        ret_s: pd.Series,
        ret_f: pd.Series,
        spot: pd.Series,
        futures: pd.Series,
        lam: float = 0.94,
    ) -> pd.DataFrame:
        """Dynamic MVHR adjusted for spot/futures price ratio: h_t* = beta_t * (S_t / F_t)."""
        n = len(ret_s)
        var_f = np.zeros(n)
        cov_sf = np.zeros(n)

        init_window = min(30, n // 4)
        var_f[0] = np.var(ret_f.iloc[:init_window], ddof=1)
        cov_sf[0] = np.cov(ret_s.iloc[:init_window], ret_f.iloc[:init_window])[0, 1]

        r_s = ret_s.to_numpy()
        r_f = ret_f.to_numpy()
        s_arr = spot.to_numpy()
        f_arr = futures.to_numpy()

        for t in range(1, n):
            var_f[t] = lam * var_f[t - 1] + (1 - lam) * (r_f[t - 1] ** 2)
            cov_sf[t] = lam * cov_sf[t - 1] + (1 - lam) * (r_s[t - 1] * r_f[t - 1])

        beta_ret = cov_sf / var_f
        h_physical = np.clip(beta_ret * (s_arr / f_arr), 0.0, 1.5)
        ann_vol_f = np.sqrt(var_f * 252)

        return pd.DataFrame(
            {"EWMA_Hedge_Ratio": h_physical, "Cond_Vol_F": ann_vol_f},
            index=ret_s.index,
        )


class AirlineHedgingBacktest:
    """Out-of-sample walk-forward hedging backtest comparing linear and option-based strategies."""

    def __init__(self, df: pd.DataFrame, cfg: SimulationConfig, warmup_days: int = 252):
        self.df = df
        self.cfg = cfg
        self.warmup = warmup_days

    def run(self) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        train = self.df.iloc[: self.warmup]
        static_beta, r2 = HedgeRatioModels.static_ols_mvhr(
            train["Ret_Spot"], train["Ret_Futures"]
        )

        ewma_df = HedgeRatioModels.ewma_mvhr(
            self.df["Ret_Spot"],
            self.df["Ret_Futures"],
            self.df["Jet_Spot"],
            self.df["Proxy_Futures"],
            lam=0.94,
        )
        full_df = pd.concat([self.df, ewma_df], axis=1)

        oos = full_df.iloc[self.warmup :]
        step = 21  # Monthly rebalancing (21 trading days)
        T_month = step / 252.0
        Q = self.cfg.monthly_gallons
        r = self.cfg.risk_free_rate

        records = []
        for start_idx in range(0, len(oos) - step, step):
            entry = oos.iloc[start_idx]
            expiry = oos.iloc[start_idx + step]

            S_0, F_0 = entry["Jet_Spot"], entry["Proxy_Futures"]
            S_T, F_T = expiry["Jet_Spot"], expiry["Proxy_Futures"]
            h_dyn = entry["EWMA_Hedge_Ratio"]
            sigma_est = entry["Cond_Vol_F"]

            budget_cost = (Q * S_0) / 1e6
            unhedged_cost = (Q * S_T) / 1e6

            # 1. Naive 1:1 Futures Hedge
            pnl_naive = (F_T - F_0) * (1.0 * Q) / 1e6
            cost_naive = unhedged_cost - pnl_naive

            # 2. Dynamic EWMA MVHR Futures Hedge
            pnl_ewma = (F_T - F_0) * (h_dyn * Q) / 1e6
            cost_ewma = unhedged_cost - pnl_ewma

            # 3. Dynamic Zero-Cost Collar (Long 3% OTM Call, Short Zero-Cost Put)
            K_call = F_0 * 1.03
            K_put, _ = Black76Engine.solve_zero_cost_collar(
                F_0, K_call, T_month, r, sigma_est
            )
            collar_payoff = max(0.0, F_T - K_call) - max(0.0, K_put - F_T)
            pnl_collar = collar_payoff * (h_dyn * Q) / 1e6
            cost_collar = unhedged_cost - pnl_collar

            records.append(
                {
                    "Date": expiry.name,
                    "Budget_Cost": budget_cost,
                    "Unhedged": unhedged_cost,
                    "Naive_1to1": cost_naive,
                    "Dynamic_MVHR": cost_ewma,
                    "ZeroCost_Collar": cost_collar,
                    "Shock_Unhedged": unhedged_cost - budget_cost,
                    "Shock_Naive": cost_naive - budget_cost,
                    "Shock_MVHR": cost_ewma - budget_cost,
                    "Shock_Collar": cost_collar - budget_cost,
                    "Hedge_Ratio": h_dyn,
                    "K_Put": K_put,
                    "K_Call": K_call,
                }
            )

        results_df = pd.DataFrame(records).set_index("Date")
        summary_df = self._compute_risk_metrics(results_df, static_beta, r2)
        return results_df, summary_df, ewma_df

    @staticmethod
    def _compute_risk_metrics(
        res: pd.DataFrame, static_beta: float, r2: float
    ) -> pd.DataFrame:
        mapping = [
            ("Unhedged", "Unhedged", "Shock_Unhedged"),
            ("Naive_1to1", "Naive_1to1", "Shock_Naive"),
            ("Dynamic_MVHR", "Dynamic_MVHR", "Shock_MVHR"),
            ("ZeroCost_Collar", "ZeroCost_Collar", "Shock_Collar"),
        ]
        unhedged_shock_var = res["Shock_Unhedged"].var(ddof=1)
        metrics = []

        for label, cost_col, shock_col in mapping:
            cost_s = res[cost_col]
            shock_s = res[shock_col]

            var_95_shock = np.percentile(shock_s, 95)
            cvar_95_shock = shock_s[shock_s >= var_95_shock].mean()
            var_red = (1.0 - shock_s.var(ddof=1) / unhedged_shock_var) * 100.0

            metrics.append(
                {
                    "Mean_Bill ($M)": round(cost_s.mean(), 2),
                    "Shock_Std ($M)": round(shock_s.std(ddof=1), 2),
                    "Var_Reduction (%)": round(var_red, 1),
                    "95%_VaR_Shock ($M)": round(var_95_shock, 2),
                    "95%_CVaR_Shock ($M)": round(cvar_95_shock, 2),
                    "Worst_Spike ($M)": round(shock_s.max(), 2),
                    "Best_Savings ($M)": round(shock_s.min(), 2),
                }
            )

        print(
            f"In-Sample OLS Return Beta: {static_beta:.3f} | "
            f"Hedge Effectiveness (R²): {r2:.3f}\n"
        )
        return pd.DataFrame(metrics, index=[m[0] for m in mapping])


def plot_hedging_dashboard(
    market_df: pd.DataFrame,
    ewma_df: pd.DataFrame,
    res_df: pd.DataFrame,
    warmup_days: int = 252,
    save_path: str = "airline_fuel_hedging_dashboard.png",
) -> None:
    fig, axes = plt.subplots(3, 1, figsize=(12, 11), constrained_layout=True)

    # Panel 1: Spot vs. Futures
    ax1 = axes[0]
    ax1.plot(market_df.index, market_df["Jet_Spot"], label="USGC Jet Fuel Spot ($/gal)", color="#1f77b4", lw=1.4)
    ax1.plot(market_df.index, market_df["Proxy_Futures"], label="Liquid Proxy Futures ($/gal)", color="#ff7f0e", lw=1.2, alpha=0.85)
    ax1.axvline(market_df.index[warmup_days], color="black", linestyle="--", alpha=0.6, label="OOS Backtest Start")
    ax1.set_title("Simulated Jet Fuel Spot vs. Proxy Futures (Jump-Diffusion + Mean-Reverting Crack Spread)", fontweight="bold")
    ax1.set_ylabel("Price ($ / Gallon)")
    ax1.legend(loc="upper left")
    ax1.grid(True, alpha=0.3)

    # Panel 2: Dynamic EWMA Hedge Ratio
    ax2 = axes[1]
    ax2.plot(ewma_df.index, ewma_df["EWMA_Hedge_Ratio"], label="Dynamic EWMA Optimal Hedge Ratio ($h_t^*$)", color="#2ca02c", lw=1.3)
    ax2.axhline(1.0, color="#d62728", linestyle=":", lw=1.2, label="Naive 1:1 Hedge ($h = 1.0$)")
    ax2.axvline(market_df.index[warmup_days], color="black", linestyle="--", alpha=0.6)
    ax2.set_title("Time-Varying Minimum Variance Hedge Ratio (RiskMetrics EWMA, $\\lambda = 0.94$)", fontweight="bold")
    ax2.set_ylabel("Hedge Ratio (Gal Futures / Gal Jet)")
    ax2.legend(loc="upper right")
    ax2.grid(True, alpha=0.3)

    # Panel 3: Monthly Budget Surprise Comparison
    ax3 = axes[2]
    ax3.plot(res_df.index, res_df["Shock_Unhedged"], label="Unhedged Budget Surprise ($M)", color="#7f7f7f", alpha=0.65, lw=1.3)
    ax3.plot(res_df.index, res_df["Shock_MVHR"], label="Dynamic MVHR Futures ($M)", color="#2ca02c", lw=1.6)
    ax3.plot(res_df.index, res_df["Shock_Collar"], label="Zero-Cost Collar ($M)", color="#9467bd", lw=1.5, linestyle="-.")
    ax3.axhline(0.0, color="black", lw=0.8)
    ax3.set_title("Out-of-Sample Monthly Fuel Budget Shock vs. Month-Start Forecast ($ Millions)", fontweight="bold")
    ax3.set_ylabel("Cost Surprise ($M)")
    ax3.set_xlabel("Date")
    ax3.legend(loc="upper left")
    ax3.grid(True, alpha=0.3)

    plt.savefig(save_path, dpi=150)
    plt.close(fig)


if __name__ == "__main__":
    config = SimulationConfig()
    market_df = MarketDataGenerator(config).generate()

    backtester = AirlineHedgingBacktest(market_df, config, warmup_days=252)
    monthly_res, risk_summary, ewma_df = backtester.run()

    print("=== Out-of-Sample Airline Fuel Hedging Performance (48 Months) ===")
    print(risk_summary.to_string())

    plot_hedging_dashboard(market_df, ewma_df, monthly_res)

In-Sample OLS Return Beta: 0.749 | Hedge Effectiveness (R²): 0.939

=== Out-of-Sample Airline Fuel Hedging Performance (48 Months) ===
                 Mean_Bill ($M)  Shock_Std ($M)  Var_Reduction (%)  95%_VaR_Shock ($M)  95%_CVaR_Shock ($M)  Worst_Spike ($M)  Best_Savings ($M)
Unhedged                  24.53            2.11                0.0                3.42                 4.06              4.84              -4.87
Naive_1to1                24.28            0.55               93.2                0.99                 1.13              1.30              -1.03
Dynamic_MVHR              24.33            0.44               95.7                0.60                 0.85              1.27              -0.89
ZeroCost_Collar           24.39            0.65               90.6                0.96                 1.19              1.38              -1.58
